# 07 — Modelo final: LightGBM global (39 municipios)

Reentrena con todos los datos (39 municipios, meses objetivo 2012–2025) el LightGBM global del notebook 05. Es el modelo que queda en `models_global/lightgbm/`.

**Regla (la misma de `metadata.json`):** `num_leaves` = moda de los 10 pliegues del 05; número de árboles y umbral de decisión = mediana de los 10 pliegues. Los valores por pliegue se leen de `hiperparametros_por_pliegue.csv`, que guarda el 05. Resultado esperado: 7 hojas, 221 árboles, umbral 0,425.

Reutiliza las secciones 1–9 del 05 (las ejecuta, no las copia), así que necesita el 05 y los datos en Drive. No escribe en las carpetas del 05: deja el modelo en `Datos/dengue/modelo_final_lightgbm/`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, hashlib, datetime, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat', 'lightgbm', 'python-calamine'], check=True)
import nbformat, numpy as np, pandas as pd, lightgbm as lgb

NB05 = os.environ.get('DENGUE_NB05', '/content/drive/MyDrive/Colab Notebooks/05_dl_profundo_dengue.ipynb')
SAL = os.environ.get('DENGUE_SALIDA_07', '/content/drive/MyDrive/Datos/dengue/modelo_final_lightgbm')
os.environ['DENGUE_SALIDA'] = '/content/salida_05_tmp'      # las celdas del 05 escriben aquí, no en sus carpetas

nb = nbformat.read(NB05, as_version=4)
C = [c.source for c in nb.cells if c.cell_type == 'code']
fin = next(i for i, s in enumerate(C) if 'def lightgbm_global' in s)
G = {'get_ipython': get_ipython, '__name__': '__main__'}
for i, s in enumerate(C[:fin + 1]):                          # secciones 1–9 del 05
    exec(compile(s, f'05_celda_{i}', 'exec'), G)
import matplotlib.pyplot as plt; plt.close('all')
assert G['FRECUENCIA'] == 'mensual' and len(G['MUNIS']) == 39
tabla, X, MUNIS = G['tabla'], G['X_LGB'], G['MUNIS']
os.makedirs(SAL, exist_ok=True)
print('lightgbm', lgb.__version__, '|', len(tabla), 'filas |', len(X), 'variables')

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Keras backend: tensorflow | RAPIDO: False
GPU: no hay (Entorno de ejecución → Cambiar tipo → GPU T4 para ir más rápido)
Leído /content/drive/MyDrive/Datos/dengue/procesado/casos_limpios.parquet
1,700,889 casos | 1,115 municipios | 2007-01-01 a 2025-12-31
Frecuencia: mensual | salida: /content/salida_05_tmp/mensual
228 periodos × 1115 municipios
39 municipios con canal estable (<20% de meses con <5 casos)
ONI descargado: 1950-02 a 2026-10
Proporción de periodos en exceso (2012+): mediana 0.33 | rango 0.16 - 0.62


[figura omitida en la copia del repositorio]

6,045 filas | 39 municipios | 2012–2025 | prevalencia de exceso 0.34 | ONI: sí
lightgbm 4.6.0 | 6045 filas | 20 variables


In [2]:
# Hiperparámetros por pliegue del 05 y reentrenamiento final
HP = f"{G['BASE']}/dengue/comparacion_dl_profundo/mensual/hiperparametros_por_pliegue.csv"
h = pd.read_csv(HP)
h = h[h['modelo'] == 'LightGBM global']
assert len(h) == 10, f'Se esperaban 10 pliegues, hay {len(h)}'
nl = int(h['num_leaves'].mode().iloc[0]); n_arb = int(round(h['arboles'].median())); umbral = float(h['umbral'].median())
print('num_leaves', nl, '| árboles', n_arb, '| umbral', umbral)

prm = {**G['LGB_BASE'], **next(g for g in G['LGB_GRID'] if g['num_leaves'] == nl)}
modelo = lgb.train(prm, lgb.Dataset(tabla[X], tabla['y'], categorical_feature=['mun_id']), num_boost_round=n_arb)
modelo.save_model(f'{SAL}/lightgbm_global_final.txt')

meta = {'modelo': 'LightGBM global, 39 municipios con canal estable, mensual',
        'entrenado': datetime.date.today().isoformat(), 'filas': len(tabla), 'periodo_objetivo': '2012-2025',
        'prevalencia_exceso': round(float(tabla['y'].mean()), 3), 'municipios': len(MUNIS),
        'mun_id': {m: i for i, m in enumerate(MUNIS)}, 'variables': X, 'horizonte_meses': 2, 'oni': True,
        'num_leaves': nl, 'arboles': n_arb, 'umbral_decision': umbral, 'hiperparametros': prm,
        'como_se_eligio': 'num_leaves = moda de los 10 pliegues del 05; árboles y umbral de decisión = mediana de los 10 pliegues',
        'uso': 'P(exceso en el mes t) = modelo.predict(X), con X construida como en la sección 7 del notebook 05 '
               '(origen t-2, mun_id según el diccionario mun_id). Alerta si P >= umbral_decision.'}
json.dump(meta, open(f'{SAL}/metadata.json', 'w'), indent=2, ensure_ascii=False)
(pd.Series(modelo.feature_importance('gain'), index=X).sort_values(ascending=False)
   .rename('ganancia').to_csv(f'{SAL}/importancia_variables.csv'))

sha = hashlib.sha256(open(f'{SAL}/lightgbm_global_final.txt', 'rb').read()).hexdigest()
print('sha256 del modelo:', sha)
print('Coincide con la regla del repo (7, 221, 0.425):', (nl, n_arb, round(umbral, 3)) == (7, 221, 0.425))
print('Guardado en', SAL)

num_leaves 7 | árboles 221 | umbral 0.42500000000000004
sha256 del modelo: db0079d3044ec43d553b0f70b81ce8e50481111ddd1c25269ff764dc8959335e
Coincide con la regla del repo (7, 221, 0.425): True
Guardado en /content/drive/MyDrive/Datos/dengue/modelo_final_lightgbm
